# SPRITE: Spatial Gene Imputation (Pseudo-6K Benchmark)

SPRITE first predicts missing genes using SPage (spatial-smoothed gene expression),
then applies spatial graph-based reinforcement and smoothing to improve predictions.

**Input**: WTX CosMx h5ad; 6K gene list  
**Output**: `half_ori_half_sprite_spage_imp_log1p.csv` — combined 6K observed + SPRITE-imputed 12K genes


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import sys
sys.path.append("/path/to/SPRITE")
import sprite.sprite as sprite

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = "/path/to/cosmx/data"
OUT_DIR  = "/path/to/cosmx/count_csv"

In [ ]:
# Load WTX CosMx data (log1p normalized)
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_I0261_I0331_anno.h5ad")
adata.obs_names = adata.obs['fov_cell_id'].tolist()
adata.obs['fov'] = adata.obs['fov'].astype(int)

# Train/test split by FOV
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs.index):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)

test_adata  = adata[test_cells, :].copy()
train_adata = adata[train_cells, :].copy()
print(f"Train: {len(train_cells):,}  |  Test: {len(test_cells):,}")

In [ ]:
# 6K gene panel (pseudo-6K: restrict test to 6K observed genes)
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()
test_adata_sub = test_adata[:, test_adata.var.index.isin(genes_6k)].copy()
test_genes = [g for g in train_adata.var_names if g not in test_adata_sub.var_names]
print(f"Observed (6K): {test_adata_sub.n_vars}  |  To impute: {len(test_genes)}")

In [ ]:
# ── Step 1: SPage initial gene expression prediction ─────────────────────────
sprite.preprocess_data(train_adata,    standardize=False, normalize=False)
sprite.preprocess_data(test_adata_sub, standardize=False, normalize=False)

sprite.predict_gene_expression(
    test_adata_sub, train_adata, test_genes,
    method="spage", n_folds=10, n_pv=10
)

In [ ]:
# ── Step 2: SPRITE spatial graph reinforcement and smoothing ──────────────────
sprite.reinforce_gene(test_adata_sub, predicted="spage_predicted_expression",
                      alpha=0.1, tol=1e-8, cv=5)
sprite.build_spatial_graph(test_adata_sub, method="fixed_radius", n_neighbors=50)
sprite.calc_adjacency_weights(test_adata_sub, method="cosine")
sprite.smooth(test_adata_sub,
              predicted="reinforced_gene_joint_spage_predicted_expression",
              alpha=0.1, tol=1e-8)

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────
spage_pred = test_adata_sub.obsm['spage_predicted_expression'].copy()
spage_pred.columns = [g.upper() for g in spage_pred.columns]

# Observed 6K counts
raw_6k = pd.DataFrame(test_adata_sub.X)
raw_6k.columns = [g.upper() for g in test_adata_sub.var_names]
raw_6k.index   = test_adata_sub.obs_names.tolist()

# Keep only imputed genes (not in 6K panel) from SPRITE predictions
imputed_cols = [g for g in spage_pred.columns if g not in [x.upper() for x in genes_6k]]
spage_pred_sub = spage_pred[imputed_cols]

# Combine: 6K observed + SPRITE-imputed 12K
combined = pd.concat([spage_pred_sub.reset_index(), raw_6k], axis=1)
combined.index = combined['index'].tolist()
combined = combined.drop('index', axis=1)

combined.to_csv(f"{OUT_DIR}/half_ori_half_sprite_spage_imp_log1p.csv")
print(f"Exported: {combined.shape[0]:,} cells x {combined.shape[1]:,} genes")